# VOS · Wan2.2 Animate 14B GGUF Q3_K_M · T4 experimental independente
**IMPLEMENTADO MAS SEM VALIDAR EM T4.** Nenhuma geração em T4/Colab gratuito foi comprovada. Este notebook não altera o baseline. Não use plano pago, Hugging Face, túnel ou API externa. ComfyUI roda como biblioteca local, sem abrir servidor. Primeiro teste: uma cena de até **1 segundo**, máscaras nativas revisadas, resolução interna reduzida. O VOS final conserva manifesto, resolução/FPS/duração; a qualidade interna pode cair.


In [ ]:
import sys,pathlib,shutil,subprocess,json,os
from google.colab import files
GGUF_ENABLED=False
"""Independent Turing gate: no BF16/32-GiB baseline waiver and no network."""
import datetime
import json
from pathlib import Path
import shutil


def diagnose(root='/content',min_disk_gib=33,torch_module=None,ram_bytes=None,probe=True):
    report={'schema':'vos-t4-gguf-report/1','profile':'LOW_VRAM_GGUF_EXPERIMENTAL',
        'status':'IMPLEMENTADO_MAS_SEM_VALIDAR_EM_T4','real_generation':False,'wan_t4_proven':False,
        'timestamp_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),
        'gpu':None,'vram_gib':0,'free_vram_gib':0,'available_ram_gib':0,'disk_free_gib':0,
        'bf16_required':False,'generator_precision':'fp16','text_precision':'fp32 (separate process)',
        'attention':'PyTorch EFFICIENT_ATTENTION SDPA, FlashAttention disabled','kernel_probe_performed':False,'nvcc_available':bool(shutil.which('nvcc') or Path('/usr/local/cuda/bin/nvcc').is_file()),'failures':[]}
    if ram_bytes is None:
        try:
            import psutil
            ram_bytes=psutil.virtual_memory().available
        except ImportError:ram_bytes=0
    report['available_ram_gib']=ram_bytes/2**30
    report['disk_free_gib']=shutil.disk_usage(root).free/2**30
    try:
        if torch_module is None:import torch as torch_module
        t=torch_module;report['torch']=str(t.__version__);report['cuda_runtime']=t.version.cuda
        if not t.cuda.is_available():raise RuntimeError('CUDA indisponivel')
        p=t.cuda.get_device_properties(0);report['gpu']=p.name
        capability=t.cuda.get_device_capability(0);report['compute_capability']=list(capability)
        report['vram_gib']=p.total_memory/2**30;report['free_vram_gib']=t.cuda.mem_get_info(0)[0]/2**30
        if capability<(7,5):report['failures'].append('GPU abaixo de capability 7.5')
        if probe:
            probe_kernels(t);report['kernel_probe_performed']=True
        else:report['failures'].append('Probe de kernels nao executado; apenas simulacao/inspecao')
    except Exception as error:report['failures'].append('GPU/CUDA/SDPA: '+str(error)[:500])
    if not report['nvcc_available']:report['failures'].append('Compilador NVCC ausente; SAM2 fixado exige extensao CUDA. Nao baixar pesos.')
    if report['vram_gib']<14 or report['free_vram_gib']<13:report['failures'].append('Exige pelo menos 14 GiB VRAM total e 13 GiB livre')
    if report['available_ram_gib']<10:report['failures'].append('RAM disponivel abaixo de 10 GiB; offload nao autorizado')
    if report['disk_free_gib']<min_disk_gib:report['failures'].append(f'Disco livre abaixo de {min_disk_gib} GiB')
    report['eligible_for_experimental_attempt']=not report['failures']
    report['warning']='Limiar de tentativa, nao minimo certificado. 8.63 GB de arquivo nao equivale ao pico de VRAM/RAM. Sem promessa de conclusao no Colab gratuito.'
    return report


def probe_kernels(t):
    # No model/weights: actual installed Turing kernels, not a GPU-name claim.
    from torch.nn.attention import sdpa_kernel,SDPBackend
    with t.inference_mode(),sdpa_kernel(SDPBackend.EFFICIENT_ATTENTION):
        for dtype in (t.float16,t.float32):
            q=t.randn(1,4,32,128,device='cuda',dtype=dtype)
            result=t.nn.functional.scaled_dot_product_attention(q,q,q)
            if not t.isfinite(result).all().item():raise RuntimeError('SDPA produziu nao-finitos')
        del q,result
    t.cuda.synchronize();t.cuda.empty_cache()


def require_attempt(report,enabled=False):
    if not enabled:raise ValueError('Perfil experimental desligado. Leia o diagnostico antes de ativar conscientemente.')
    if not report.get('eligible_for_experimental_attempt') or not report.get('kernel_probe_performed'):
        raise ValueError('Diagnostico GGUF bloqueado: '+ '; '.join(report.get('failures',[])))

REPORT=diagnose('/content',min_disk_gib=33)
pathlib.Path('/content/low_vram_report.json').write_text(json.dumps(REPORT,ensure_ascii=False,indent=2))
print(json.dumps(REPORT,ensure_ascii=False,indent=2))
if not REPORT['eligible_for_experimental_attempt']:
    print('BLOQUEADO. Guarde low_vram_report.json. Nenhum peso/video foi baixado. Nao contorne o diagnostico.')
    try:files.download('/content/low_vram_report.json')
    except Exception:pass
else:print('Hardware permite uma tentativa experimental; nao comprova o modelo. Leia e ative a proxima celula se desejar tentar.')

## 1. Ativação consciente e ferramentas fixadas
O perfil começa desligado. Só ative depois de um diagnóstico elegível com kernels CUDA/SDPA realmente verificados. 8,63 GB é tamanho do arquivo, não consumo total. Exige RAM e disco livres; se não houver recursos gratuitos, pare.


In [ ]:
GGUF_ENABLED=False
LOW_VRAM_GGUF_EXPERIMENTAL=False #@param {type:"boolean"}
require_attempt(REPORT,LOW_VRAM_GGUF_EXPERIMENTAL)
GGUF_ENABLED=True
VOS_REF='b0369fdc193784e5024edec2203fa6fea2b10a38'
ROOT=pathlib.Path('/content/vos_t4_gguf');ROOT.mkdir(exist_ok=True)
TOOLS=ROOT/'tools/frame_studio';(TOOLS/'experimental').mkdir(parents=True,exist_ok=True);(TOOLS/'static').mkdir(exist_ok=True)
import urllib.request
for name in ('__init__.py','wan_bridge.py','target_masks.py','wan_target.py','colab_support.py','recovery.py','static/target_editor.js','experimental/__init__.py','experimental/gguf_profile.py','experimental/gguf_probe.py','experimental/gguf_t4.py','experimental/gguf_engine.py'):
    urllib.request.urlretrieve(f'https://raw.githubusercontent.com/aruanmf446-hub/Vos/{VOS_REF}/frame_studio/{name}',str(TOOLS/name))
sys.path.insert(0,str(TOOLS.parent))
from frame_studio.colab_support import Session,Pipeline,install_error_hook
from frame_studio.experimental import gguf_t4 as experiment
from frame_studio.experimental.gguf_profile import diagnose as fresh_diagnose
from frame_studio.target_masks import show_editor,require_review
from frame_studio.recovery import export_progress
MANIFEST=ROOT/'wan22_t4_gguf_manifest.json'
urllib.request.urlretrieve(f'https://raw.githubusercontent.com/aruanmf446-hub/Vos/{VOS_REF}/colab/wan22_t4_gguf_manifest.json',str(MANIFEST))
DATA=experiment.manifest(MANIFEST)
SESSION=Session(ROOT/'logs',download=files.download);FLOW=Pipeline(SESSION);install_error_hook(SESSION,get_ipython())
SESSION.folder.joinpath('low_vram_report.json').write_text(json.dumps(REPORT,indent=2))
# Preserve the specific profile diagnosis in bounded/sanitized failure ZIPs.
def save_gguf_failure(result):
    if getattr(result,'error_in_exec',None) or getattr(result,'error_before_exec',None):
        REPORT['status']='FALHOU_SEM_COMPROVACAO_T4';REPORT['real_generation']=False
        REPORT['interrupted_stage']=SESSION.current_stage
        SESSION.folder.joinpath('low_vram_report.json').write_text(json.dumps(REPORT,indent=2))
        files.download(str(SESSION.bundle()))
get_ipython().events.register('post_run_cell',save_gguf_failure)
INSTALL_OK=False;WEIGHTS_OK=False


## 2. Instalar motores e testar imports/kernels antes dos pesos
Revisões Git fixadas, Torch/CUDA do Colab preservados. Sem FlashAttention2, SageAttention ou xFormers. SAM2 fixado requer compilador NVCC. Esta instalação ainda exige validação real no Colab. Erro bloqueia downloads de pesos; guarde VOS_LOGS.zip.


In [ ]:
if not globals().get('GGUF_ENABLED',False):raise RuntimeError('Perfil experimental desligado ou diagnostico bloqueado. Nao continuar.')
INSTALL_OK=False;WEIGHTS_OK=False
FLOW.begin('install')
experiment.install(ROOT,DATA,REPORT,SESSION,enabled=GGUF_ENABLED)
REPORT=fresh_diagnose('/content',min_disk_gib=33)
require_attempt(REPORT,GGUF_ENABLED)
(ROOT/'low_vram_report.json').write_text(json.dumps(REPORT,indent=2))
SESSION.folder.joinpath('low_vram_report.json').write_text(json.dumps(REPORT,indent=2))
INSTALL_OK=True
print('Imports e kernels do motor passaram; isso ainda nao e geracao nem prova de memoria do modelo.')

## 3. Baixar apenas os arquivos permitidos pelo ModelScope
Cerca de 16,9 GiB selecionados: DiT Q3_K_M, UMT5 GGUF, CLIP, VAE, relighting e pré-processamento. Reserva adicional de 16 GiB para pacotes/intermediários. Tamanho e SHA-256 são obrigatórios para todos. Sem snapshot BF16 integral, sem espelhos. O VAE armazenado em BF16 é convertido para FP32 em processo separado; o DiT calcula em FP16.


In [ ]:
if not globals().get('GGUF_ENABLED',False):raise RuntimeError('Perfil experimental desligado ou diagnostico bloqueado. Nao continuar.')
if not INSTALL_OK:raise RuntimeError('Instalacao anterior nao concluiu. Nao baixar pesos.')
WEIGHTS_OK=False
experiment.download_weights(ROOT,DATA,REPORT,enabled=GGUF_ENABLED)
WEIGHTS_OK=True;FLOW.complete('install')
print('Arquivos verificados. Agora crie um projeto NOVO e curto no VOS; Render e temporario.')

## 4. Preparar ZIP VOS — uma cena de até 1 segundo, Original
Guarde ZIP original e referência no computador. Mesmo projeto/sessão Render no retorno: reinício/hibernação/TTL podem apagar o projeto; este notebook não impede isso. Não envie vídeo privado antes de preparar o runtime.


In [ ]:
if not globals().get('GGUF_ENABLED',False):raise RuntimeError('Perfil experimental desligado ou diagnostico bloqueado. Nao continuar.')
if not WEIGHTS_OK:raise RuntimeError('Pesos nao verificados.')
FLOW.begin('prepare',requires=('install',))
print('Escolha ZIP VOS original, ate 1 segundo')
u=files.upload();ZIP=pathlib.Path('/content')/next(n for n in u if n.lower().endswith('.zip'));del u
print('Escolha referencia do novo personagem')
u=files.upload();REF=pathlib.Path('/content')/next(n for n in u if n.lower().endswith(('.png','.jpg','.jpeg')));del u
import tempfile
WORK=pathlib.Path(tempfile.mkdtemp(prefix='job-',dir=ROOT));SESSION.work=WORK
BRIDGE=TOOLS/'wan_bridge.py'
SESSION.run([sys.executable,str(BRIDGE),'prepare','--zip',str(ZIP),'--reference',str(REF),'--work',str(WORK)],label='VOS preparar nativo')
from fractions import Fraction
if Fraction(json.loads((WORK/'manifest.json').read_text())['selection']['duration'])>1:raise RuntimeError('Exporte uma cena de ate 1 segundo para este primeiro experimento.')
FLOW.complete('prepare')
from IPython.display import display,Video
display(Video(str(WORK/'driving.mp4'),embed=True))

## 5. Selecionar o alvo e guardar pontos
Reutiliza o editor VOS existente: verde no alvo, vermelho na outra pessoa. Não há reconhecimento semântico do nome. Aqui a seleção explícita é obrigatória, para não seguir automaticamente o maior personagem.


In [ ]:
if not globals().get('GGUF_ENABLED',False):raise RuntimeError('Perfil experimental desligado ou diagnostico bloqueado. Nao continuar.')
FLOW.begin('selection',requires=('prepare',))
show_editor(WORK,'prompts')
files.download(str(export_progress(WORK,ROOT/'VOS_T4_PROGRESSO_pontos.zip')))
FLOW.complete('selection')

## 6. SAM2 FP16 isolado e revisão nativa
Só SAM2 fica carregado neste processo. FP16 e atenção matemática compatível, sem FlashAttention2. Ainda não comprovado em T4. Revise/corrija cada máscara; alertas bloqueiam aprovação. Pontos e máscaras são os mesmos do fluxo VOS. Guarde o backup antes do modelo generativo.


In [ ]:
if not globals().get('GGUF_ENABLED',False):raise RuntimeError('Perfil experimental desligado ou diagnostico bloqueado. Nao continuar.')
FLOW.begin('track',requires=('selection','install'))
SESSION.run([sys.executable,str(TOOLS/'wan_target.py'),'--precision','fp16','track','--work',str(WORK),'--wan',str(ROOT/'Wan2.2'),'--checkpoint',str(ROOT/'weights')],label='SAM2 FP16 isolado experimental')
show_editor(WORK,'review')
files.download(str(export_progress(WORK,ROOT/'VOS_T4_PROGRESSO_mascaras.zip')))
FLOW.complete('track')

## 7. Pose/rosto e revisão dos intermediários
SAM2 já saiu da memória. O pré-processamento existente usa somente o alvo e mantém o cenário fora da máscara. Confira pessoa, pose, rosto e máscara em toda a cena. SIM registra revisão humana; não corrige erro de rastreamento.


In [ ]:
if not globals().get('GGUF_ENABLED',False):raise RuntimeError('Perfil experimental desligado ou diagnostico bloqueado. Nao continuar.')
FLOW.begin('preprocess',requires=('track','install'))
SESSION.run([sys.executable,str(BRIDGE),'preprocess','--work',str(WORK),'--wan',str(ROOT/'Wan2.2'),'--checkpoint',str(ROOT/'weights')],label='Pose rosto VOS isolados')
FLOW.complete('preprocess');FLOW.begin('review',requires=('preprocess',))
for name in ('src_mask.mp4','src_pose.mp4','src_face.mp4'):
    print(name);display(Video(str(WORK/'wan_processed'/name),embed=True))
if input('Apenas o alvo correto em todos os quadros? Digite SIM: ').strip().upper()!='SIM':raise RuntimeError('Cancelado; corrija selecao e mascaras.')
SESSION.run([sys.executable,str(BRIDGE),'review-mask','--work',str(WORK),'--confirm','SIM'],label='Revisao humana de intermediarios')
FLOW.complete('review')

## 8. Primeira inferência real GGUF — tentativa experimental
Texto, condicionamento, amostragem e decode rodam em quatro processos separados. DiT FP16/SDPA eficiente, 30 blocos CPU, LoRA sem merge, sem compilação, VAE FP32 tiled isolado. Internamente até 320 px, com letterbox; o `finish` original recompõe resolução nativa e pixels fora da máscara. Padding de condição até 33 quadros repete o último e só a cauda extra é removida. Não há interpolação. Um resultado não-finito/incompleto ou contagem diferente aborta.


In [ ]:
if not globals().get('GGUF_ENABLED',False):raise RuntimeError('Perfil experimental desligado ou diagnostico bloqueado. Nao continuar.')
FLOW.begin('infer',requires=('install','review'))
REPORT=fresh_diagnose('/content',min_disk_gib=8)
require_attempt(REPORT,GGUF_ENABLED)
experiment.run_generation(WORK,ROOT,DATA,REPORT,SESSION,enabled=GGUF_ENABLED)
REPORT['real_generation']=True
REPORT['wan_t4_proven']=False # exige validacao visual do usuario e evidencia do hardware
REPORT['status']='GERACAO_PRODUZIDA_AGUARDA_VALIDACAO_VISUAL'
REPORT['engine_receipt']=json.loads((WORK/'gguf_engine_receipt.json').read_text())
(ROOT/'low_vram_report.json').write_text(json.dumps(REPORT,indent=2))
SESSION.folder.joinpath('low_vram_report.json').write_text(json.dumps(REPORT,indent=2))
FLOW.complete('infer');display(Video(str(WORK/'wan_generated.mp4'),embed=True))

## 9. ZIP editado pelo finish VOS existente
Só depois de geração real com recibo e contagem/FPS válidos. Importe apenas VOS_EDITADO.zip no mesmo projeto VOS original. Backup de máscaras e logs não são ZIPs de importação. Se Render expirou, não altere manifesto para forçar importação.


In [ ]:
if not globals().get('GGUF_ENABLED',False):raise RuntimeError('Perfil experimental desligado ou diagnostico bloqueado. Nao continuar.')
FLOW.begin('finish',requires=('infer',))
OUT=ROOT/'VOS_EDITADO.zip'
SESSION.run([sys.executable,str(BRIDGE),'finish','--work',str(WORK),'--output',str(OUT)],label='Recompor ZIP VOS original')
FLOW.complete('finish')
files.download(str(OUT));files.download(str(ROOT/'low_vram_report.json'));files.download(str(SESSION.bundle()))